# Notebook 04 — QLoRA Fine-Tuning (SFT) with Unsloth

**Model:** `Qwen/Qwen2.5-3B-Instruct`  
**Method:** 4-bit QLoRA via Unsloth + TRL SFTTrainer  
**GPU:** T4 (15 GB VRAM)

> **Prerequisites:** NB01 và NB02 đã chạy xong.

## 0. Mount Drive

In [1]:
from pathlib import Path
import os

if os.path.exists('/content/'):
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NLP_Final/Source/')
else:
    BASE = Path('../')

print(f'Base: {BASE}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base: /content/drive/MyDrive/NLP_Final/Source


## 1. Install

In [2]:
%%capture
!pip install unsloth -q
!pip install -U trl peft accelerate datasets huggingface-hub python-dotenv -q

## 2. GPU Check

In [3]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU not available.')

total_vram    = torch.cuda.get_device_properties(0).total_memory / 1e9
COMPUTE_DTYPE = torch.bfloat16 if total_vram > 30 else torch.float16

print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {total_vram:.1f} GB')
print(f'dtype: {COMPUTE_DTYPE}')

GPU  : Tesla T4
VRAM : 15.6 GB
dtype: torch.float16


## 3. Load & Format Training Data

In [4]:
import json

SYSTEM_PROMPT = (
    'Ban la tro ly AI cua Truong Dai hoc Ton Duc Thang (TDTU). '
    'Ban tra loi cac cau hoi ve quy che, quy dinh, chinh sach cua truong '
    'mot cach chinh xac va day du. '
    'Tra loi bang tieng Viet. Neu khong co du thong tin, hay noi ro dieu do.'
)


def format_chatml(pair: dict) -> str:
    return (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{pair["question"]}<|im_end|>\n'
        f'<|im_start|>assistant\n{pair["answer"]}<|im_end|>'
    )


def format_chatml_multi_turn(conv: dict) -> str:
    text = f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
    for turn in conv['turns']:
        text += f'<|im_start|>{turn["role"]}\n{turn["content"]}<|im_end|>\n'
    return text.rstrip('\n')


train_pairs, test_pairs = [], []
with open(BASE / 'data/qa_filtered/qa_train.jsonl', encoding='utf-8') as f:
    for line in f:
        if line.strip(): train_pairs.append(json.loads(line))
with open(BASE / 'data/qa_filtered/qa_test.jsonl', encoding='utf-8') as f:
    for line in f:
        if line.strip(): test_pairs.append(json.loads(line))

verified = [p for p in train_pairs if p.get('human_verified')]
pool     = verified if verified else train_pairs
train_texts = [format_chatml(p) for p in pool]
print(f'Train QA pairs: {len(train_texts)} ({"human-verified" if verified else "all"})')

verified_test = [p for p in test_pairs if p.get('human_verified')]
eval_texts    = [format_chatml(p) for p in (verified_test if verified_test else test_pairs)]

conv_texts = []
conv_path  = BASE / 'data/qa_filtered/conversations_train.jsonl'
if conv_path.exists():
    with open(conv_path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                c = json.loads(line)
                if c.get('turns') and len(c['turns']) >= 4:
                    conv_texts.append(format_chatml_multi_turn(c))
    print(f'Multi-turn: {len(conv_texts)}')

all_train_texts = train_texts + conv_texts
print(f'Total train : {len(all_train_texts)} | Eval: {len(eval_texts)}')

Train QA pairs: 1174 (human-verified)
Multi-turn: 135
Total train : 1309 | Eval: 264


## 4. Load Model (Unsloth)

In [7]:
import os
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

from unsloth import FastLanguageModel
from dotenv import load_dotenv

load_dotenv(BASE / '.env')
HF_TOKEN = os.environ.get('HF_TOKEN')

MODEL_ID       = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = COMPUTE_DTYPE,
    load_in_4bit   = True,
    token          = HF_TOKEN,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Loaded: {MODEL_ID}')
print(f'VRAM  : {torch.cuda.memory_allocated(0)/1e9:.2f} GB')

==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded: Qwen/Qwen2.5-3B-Instruct
VRAM  : 2.40 GB


## 5. QLoRA Config

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.0,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable/1e6:.2f}M / {total/1e6:.2f}M ({100*trainable/total:.2f}%)')

Unsloth 2026.5.2 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Trainable: 29.93M / 1830.06M (1.64%)


## 6. Build Dataset

In [9]:
from datasets import Dataset

def truncate(texts: list[str], max_tok: int) -> list[str]:
    out = []
    for t in texts:
        ids = tokenizer.encode(t, add_special_tokens=False)
        if len(ids) > max_tok:
            t = tokenizer.decode(ids[:max_tok], skip_special_tokens=False)
        out.append(t)
    return out

train_dataset = Dataset.from_dict({'text': truncate(all_train_texts, MAX_SEQ_LENGTH)})
eval_dataset  = Dataset.from_dict({'text': truncate(eval_texts,      MAX_SEQ_LENGTH)})

print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

Train: 1309 | Eval: 264


## 7. Train

`packing=True` → Unsloth's `padding_free` guard không trigger  
(guard chỉ kích hoạt khi `packing=False AND max_length is not None`)

In [10]:
from trl import SFTConfig, SFTTrainer

SFT_OUTPUT = str(BASE / 'models/sft_checkpoint')

training_args = SFTConfig(
    output_dir                  = SFT_OUTPUT,
    num_train_epochs            = 3,
    per_device_train_batch_size = 2,
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = 8,
    learning_rate               = 2e-4,
    warmup_steps                = 7,
    lr_scheduler_type           = 'cosine',
    optim                       = 'adamw_8bit',
    fp16                        = (COMPUTE_DTYPE == torch.float16),
    bf16                        = (COMPUTE_DTYPE == torch.bfloat16),
    logging_steps               = 10,
    save_strategy               = 'epoch',
    eval_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    max_grad_norm               = 0.3,
    dataloader_num_workers      = 0,
    report_to                   = 'none',
    gradient_checkpointing      = True,
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = 'text',
    packing                     = True,    # bypass padding_free guard
    packing_strategy            = 'bfd',
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
)

print(f'VRAM before: {torch.cuda.memory_allocated(0)/1e9:.2f} GB')
print('Training...')

result = trainer.train()

print(f'Loss    : {result.training_loss:.4f}')
print(f'Runtime : {result.metrics["train_runtime"]:.0f}s')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1309 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/1309 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/264 [00:00<?, ? examples/s]

Unsloth: Packing eval dataset (num_proc=6):   0%|          | 0/264 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 331 | Num Epochs = 3 | Total steps = 63
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss
1,1.174137,0.997812
2,0.962979,0.893402
3,0.884862,0.877163


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint/checkpoint-21/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint/checkpoint-42/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint/checkpoint-63/tokenizer_config.json.


Loss    : 1.1871
Runtime : 1388s


## 8. Save Adapter

In [11]:
trainer.save_model(SFT_OUTPUT)
tokenizer.save_pretrained(SFT_OUTPUT)
print(f'Saved -> {SFT_OUTPUT}')

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint/tokenizer_config.json.


Saved -> /content/drive/MyDrive/NLP_Final/Source/models/sft_checkpoint


## 9. Push to HuggingFace Hub

In [12]:
from huggingface_hub import login

HF_USERNAME = os.environ.get('HF_USERNAME')
REPO_NAME   = 'tdtu-student-regulations-qa-model'

if HF_TOKEN and HF_USERNAME:
    login(token=HF_TOKEN)
    trainer.model.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')
    tokenizer.push_to_hub(f'{HF_USERNAME}/{REPO_NAME}')
    print(f'Pushed -> https://huggingface.co/{HF_USERNAME}/{REPO_NAME}')
else:
    print('Set HF_TOKEN + HF_USERNAME in .env to push.')

Repo card metadata block was not found. Setting CardData to empty.


Saved model to https://huggingface.co/hungminhss/tdtu-student-regulations-qa-model


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpzesjo85d/tokenizer_config.json.


Pushed -> https://huggingface.co/hungminhss/tdtu-student-regulations-qa-model


## 10. Inference Test

In [13]:
FastLanguageModel.for_inference(model)
model.eval()


def answer(question: str, max_new_tokens: int = 256) -> str:
    prompt = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{question}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample      = False,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = tokenizer.convert_tokens_to_ids('<|im_end|>'),
        )
    gen = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


q = test_pairs[5]['question'] if test_pairs else 'Dieu kien xet hoc bong la gi?'
print(f'Q: {q}')
print(f'A: {answer(q)}')

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: Nếu em gặp vấn đề bất thường trên mạng liên quan đến trường, em nên làm gì?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


A: Dạ vâng! Nếu em phát hiện thông tin sai sự thật hoặc có dấu hiệu vi phạm pháp luật, em cần báo ngay cho phòng Công tác học sinh - sinh viên hoặc Phòng An ninh - Pháp chế. Ngoài ra, em cũng có thể phản ánh qua email: [email protected] hoặc số điện thoại: (028) 37755090. Em nhớ giữ gìn thông tin cá nhân và không chia sẻ thông tin giả mạo nhé!


---
**Output:** `models/sft_checkpoint/` (LoRA adapter)  
**Next:** `05_experiments_eval.ipynb`